In [4]:
# ============================================================
# INTERFACE D’EXPLORATION DE CORPUS – TD7 → TD10
# ============================================================
# ⚠️ À exécuter dans JUPYTER NOTEBOOK
# ============================================================

# -----------------------------
# IMPORTS
# -----------------------------
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output
from collections import Counter
import matplotlib.pyplot as plt

from Document import Document
from Corpus import Corpus
from SearchEngine import SearchEngine


# ============================================================
# 1. CHARGEMENT DU CORPUS (TD8)
# ============================================================

df = pd.read_csv("corpus.csv")

corpus = Corpus("Corpus TD")

for i, row in df.iterrows():
    auteur = row["auteur"]
    texte = row["texte"]

    phrases = texte.split(".")
    for p in phrases:
        p = p.strip()
        if p:
            doc = Document(
                titre="Phrase",
                auteur=auteur,
                date="2024-01-01",
                url="",
                texte=p
            )
            corpus.ajouter_document(doc)

print("✔ Corpus chargé :", corpus.ndoc, "phrases")


# ============================================================
# 2. MOTEUR DE RECHERCHE (TD7)
# ============================================================

moteur = SearchEngine(corpus)
print("✔ Moteur de recherche prêt")


# ============================================================
# 3. WIDGETS DE L’INTERFACE (TD8 / TD9)
# ============================================================

# Auteurs
liste_auteurs = sorted(set(doc.auteur for doc in corpus.id2doc.values()))
liste_auteurs.insert(0, "Tous")

dropdown_auteur = widgets.Dropdown(
    options=liste_auteurs,
    description="Auteur :"
)

# Recherche
champ_requete = widgets.Text(
    description="Mots-clés :",
    placeholder="ex : python apprentissage"
)

slider_k = widgets.IntSlider(
    value=5,
    min=1,
    max=30,
    description="Résultats :"
)

bouton_recherche = widgets.Button(
    description="Rechercher",
    button_style="primary"
)

# Concordancier
champ_concorde = widgets.Text(
    description="Mot :",
    placeholder="ex : python"
)

bouton_concorde = widgets.Button(
    description="Concordance",
    button_style="warning"
)

# Statistiques / Graphiques
bouton_stats = widgets.Button(
    description="📊 Statistiques",
    button_style="info"
)

bouton_graph = widgets.Button(
    description="📈 Graphique",
    button_style="success"
)

zone_sortie = widgets.Output()


# ============================================================
# 4. FONCTIONS MÉTIER
# ============================================================

def lancer_recherche(b):
    with zone_sortie:
        clear_output()

        requete = champ_requete.value.strip()
        k = slider_k.value
        auteur_filtre = dropdown_auteur.value

        if not requete:
            print("⚠️ Veuillez entrer des mots-clés.")
            return

        resultats = moteur.search(requete, k=1000)

        if auteur_filtre != "Tous":
            resultats = [d for d in resultats if d.auteur == auteur_filtre]

        resultats = resultats[:k]

        print(f"🔎 Résultats pour : « {requete} »")
        print(f"Auteur : {auteur_filtre}")
        print("-" * 60)

        if not resultats:
            print("❌ Aucun résultat trouvé.")
            return

        for doc in resultats:
            print("Auteur :", doc.auteur)
            print("Phrase :", doc.texte)
            print("-" * 60)

        # Export CSV (TD10)
        pd.DataFrame(
            [{"auteur": d.auteur, "texte": d.texte} for d in resultats]
        ).to_csv("resultats_recherche.csv", index=False)

        print("💾 Résultats exportés dans resultats_recherche.csv")


def afficher_concorde(b):
    with zone_sortie:
        clear_output()

        mot = champ_concorde.value.strip()
        if not mot:
            print("⚠️ Entrez un mot.")
            return

        print(f"📖 Concordancier pour « {mot} »")
        print("-" * 60)

        res = corpus.concorde(mot)
        if not res:
            print("❌ Mot non trouvé.")
            return

        for g, m, d in res:
            print(g, m, d)
            print("-" * 60)


def afficher_stats(b):
    with zone_sortie:
        clear_output()

        auteurs = Counter(doc.auteur for doc in corpus.id2doc.values())

        print("📊 Statistiques du corpus")
        print("-" * 50)
        print("Nombre total de phrases :", corpus.ndoc)
        print("Nombre d’auteurs :", len(auteurs))
        print("\nDocuments par auteur :")

        for a, n in auteurs.items():
            print(f" - {a} : {n}")


def afficher_graphique(b):
    with zone_sortie:
        clear_output()

        mots = []
        for doc in corpus.id2doc.values():
            mots.extend(doc.texte.lower().split())

        freq = Counter(mots).most_common(10)
        if not freq:
            print("Pas assez de données.")
            return

        labels, values = zip(*freq)

        plt.figure()
        plt.bar(labels, values)
        plt.xticks(rotation=45)
        plt.title("Mots les plus fréquents du corpus")
        plt.show()


# ============================================================
# 5. LIAISON DES BOUTONS
# ============================================================

bouton_recherche.on_click(lancer_recherche)
bouton_concorde.on_click(afficher_concorde)
bouton_stats.on_click(afficher_stats)
bouton_graph.on_click(afficher_graphique)


# ============================================================
# 6. AFFICHAGE FINAL DE L’INTERFACE
# ============================================================

ui = widgets.VBox([
    widgets.Label("🔍 Interface d’exploration de corpus (TD7–TD10)"),
    widgets.HBox([champ_requete, slider_k]),
    dropdown_auteur,
    bouton_recherche,
    widgets.HBox([champ_concorde, bouton_concorde]),
    widgets.HBox([bouton_stats, bouton_graph]),
    zone_sortie
])

display(ui)


✔ Corpus chargé : 3 phrases
✔ Moteur de recherche prêt
